# agent_framework_openai

## Example 1: `agent_framework.openai` works now (it re-exports this package)

In [ ]:
# Example 1: before this package shipped, `import agent_framework.openai` names failed; now they are this package's
import importlib.metadata
import agent_framework.openai as via_core
import agent_framework_openai as direct

assert via_core.OpenAIChatClient is direct.OpenAIChatClient
assert via_core.OpenAIChatCompletionClient is direct.OpenAIChatCompletionClient
print("Example 1:", importlib.metadata.version("agent-framework-openai"), "re-exported through agent_framework.openai")

## Example 2: build the Responses and Chat Completions clients with explicit settings

In [ ]:
# Example 2: the model and base URL given to the constructor are what the client reports
from agent_framework_openai import OpenAIChatClient, OpenAIChatCompletionClient

responses = OpenAIChatClient(model="gpt-4o-mini", api_key="sk-test", base_url="http://127.0.0.1:9/v1")
completions = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key="sk-test", base_url="http://127.0.0.1:9/v1")
for client in (responses, completions):
    assert client.model == "gpt-4o-mini", client.model
    assert str(client.base_url).startswith("http://127.0.0.1:9"), client.base_url
print("Example 2:", type(responses).__name__, "and", type(completions).__name__, "for", responses.model)

## Example 3: settings come from OPENAI_* environment variables

In [ ]:
# Example 3: with no arguments, the client reads its key and model from the environment
import os
from unittest import mock
from agent_framework_openai import OpenAIChatCompletionClient

with mock.patch.dict(os.environ, {"OPENAI_API_KEY": "sk-from-env", "OPENAI_CHAT_COMPLETION_MODEL": "gpt-env-model"}):
    client = OpenAIChatCompletionClient()
assert client.model == "gpt-env-model", client.model
print("Example 3: model from the environment:", client.model)

## Example 4: a request to a closed port fails with the framework's own exception

In [ ]:
# Example 4: nothing listens on the port, so get_response raises ChatClientException (no hang, no raw socket error)
import socket
from agent_framework import Message
from agent_framework.exceptions import ChatClientException
from agent_framework_openai import OpenAIChatCompletionClient

with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0)); port = probe.getsockname()[1]
client = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key="sk-test", base_url=f"http://127.0.0.1:{port}/v1")
try:
    await client.get_response([Message(role="user", contents=["hello"])])
except ChatClientException as error:
    print("Example 4: refused as expected:", type(error).__name__, str(error)[:80])
else:
    raise AssertionError("a request to a closed port succeeded")

## Example 5: the client plugs into an Agent with a tool

In [ ]:
# Example 5: an Agent wraps the OpenAI client and a @tool; nothing is sent until the agent runs
from agent_framework import Agent, tool
from agent_framework_openai import OpenAIChatClient

@tool(description="Return the weather for a city.")
def weather(city: str) -> str:
    return f"sunny in {city}"

agent = Agent(client=OpenAIChatClient(model="gpt-4o-mini", api_key="sk-test"), name="weather_bot",
              instructions="Answer weather questions.", tools=[weather])
assert agent.name == "weather_bot" and agent.client.model == "gpt-4o-mini"
print("Example 5:", agent.name, "with", type(agent.client).__name__, "and tool", weather.name)